In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
import pandas as pd
import scipy as sp


from scipy.stats import t
from cherab.core.atomic import carbon, hydrogen
from cherab.openadas import OpenADAS
from Emissivity_Profile_TCABR.Emissivity import get_emissivity
from abel_inversion.abel_inversion import onion_peeling_method

In [1]:
def Vphi_model(t, DeltaV, gama, a=.18):
    """Calcula o modelo de sino da velocidade toroidal eliminando os valores imaginários e divisões por zero

    Parâmetros:
        t (float): Intervalos de valores de raio (m)
        DeltaV (float): Amplitude da velocidade (m/s)
        Gama (float):  Expoente efetivo
        a (float): raio maior do tokamak (m)
    Retorna:
        V_phi (arr): V_phi(t) simulada seguindo modelo descrito em Wesson J., Tokamaks (2004).
    """
    # Recusa valores de a negativos e iguais a 0
    if a <= 0:
        return print("Please insert the original Radius of your Tokamak")

    # Adapta o código para listas e números
    t = np.asarray(t)

    # Tratamento de possíveis valores negativos
    power = gama - 1
    base = 1 - (t/a)**2

    resultado = DeltaV * gama * (2 * (t / a**2)) * (base**power)
    
    if power < 0:
        invalido = (base<=0)
    else:
        invalido = base<0
    
    return np.where(invalido,0.0,resultado)

def Vapp(eps, Vphi, W_ij):
    """Aplicação do modelo direto propagando a velocidade de plasma.

    Parâmetros:
        t (float): Valores de emissividade ao longo do raio.
        Vphi (float): Valores de velocidade toroidal ao longo do raio.
        W_ij (arr): Matriz de peso calculada utilizando o método de Onion Peeling.
    Retorna: 
        Vapp (arr): Curvas de valores de velocidade aparente.    
    """
    U = W_ij @ (eps*Vphi)
    S = W_ij @ eps
    return U/S